In [ ]:
# preprocess_wikitext2_full.py
import json
import random
from datasets import load_dataset
from transformers import BertTokenizerFast
import nltk
from nltk.tokenize import sent_tokenize

def make_nsp_pairs(sentences, num_samples):
    # Positive pairs: consecutive sentences (use all available)
    positive_pairs = []
    for i in range(len(sentences) - 1):
        positive_pairs.append((sentences[i], sentences[i + 1], 1))
    
    # Negative pairs: random non-consecutive (match num_pos)
    negative_pairs = []
    num_pos = min(len(positive_pairs), num_samples // 2)
    used_pairs = set()
    while len(negative_pairs) < num_pos:
        i = random.randint(0, len(sentences) - 2)
        j = random.randint(0, len(sentences) - 1)
        pair = (sentences[i], sentences[j], 0)
        if j != i + 1 and pair not in used_pairs:
            negative_pairs.append(pair)
            used_pairs.add(pair)
    
    # Balance: 50% pos + 50% neg, shuffle (cap at num_samples)
    all_pairs = positive_pairs[:num_pos] + negative_pairs[:num_pos]
    random.shuffle(all_pairs)
    return all_pairs[:num_samples]

def apply_mlm_masking(input_ids, tokenizer, labels):
    # Copy for labels
    labels = input_ids.copy()
    num_tokens = len(input_ids)
    # Sample 15% interior tokens (skip first/last for [CLS]/[SEP]s)
    interior_len = num_tokens - 2
    if interior_len <= 0:
        return input_ids, labels
    num_mask = max(1, int(0.15 * interior_len))
    masked_indices = random.sample(range(1, num_tokens - 1), num_mask)
    for idx in masked_indices:
        labels[idx] = input_ids[idx]  # Original for loss
        rand = random.random()
        if rand < 0.8:
            input_ids[idx] = tokenizer.mask_token_id  # [MASK]
        elif rand < 0.9:
            input_ids[idx] = random.randint(0, tokenizer.vocab_size - 1)  # Random
        # else: unchanged
    # Set non-masked to -100 (ignore in loss)
    for i in range(len(labels)):
        if i not in masked_indices:
            labels[i] = -100
    return input_ids, labels

def main():
    # Download NLTK resource if missing
    try:
        nltk.data.find('tokenizers/punkt_tab')
    except LookupError:
        print("Downloading punkt_tab for sentence tokenization...")
        nltk.download('punkt_tab')
    
    # Load full WikiText-2-v1 (standard raw version)
    dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
    tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
    
    # Extract full text and split into clean sentences
    full_text = ' '.join([doc for doc in dataset['text'] if doc.strip()])
    sentences = [s.strip() for s in sent_tokenize(full_text) if len(s.strip()) > 10]
    
    print(f"Extracted {len(sentences)} sentences from full WikiText-2-v1 train split.")
    
    # Create balanced NSP pairs (100k total for full dataset use, adjustable)
    num_samples = 100000
    pairs = make_nsp_pairs(sentences, num_samples)
    processed = []
    
    for sent_a, sent_b, is_next in pairs:
        # Tokenize with special tokens: [CLS] A [SEP] B [SEP]
        tokenized = tokenizer(sent_a, sent_b, add_special_tokens=True, max_length=128, truncation=True, 
                              return_token_type_ids=True, return_attention_mask=True)
        input_ids = tokenized['input_ids']
        token_type_ids = tokenized['token_type_ids']
        attention_mask = tokenized['attention_mask']
        
        # Apply MLM masking
        masked_input_ids, mlm_labels = apply_mlm_masking(input_ids, tokenizer, [])
        
        # Padding (ensure exactly 128)
        pad_len = 128 - len(masked_input_ids)
        if pad_len > 0:
            masked_input_ids += [tokenizer.pad_token_id] * pad_len
            mlm_labels += [-100] * pad_len
            token_type_ids += [0] * pad_len  # Pad with segment 0
            attention_mask += [0] * pad_len
        
        processed.append({
            'input_ids': masked_input_ids,
            'token_type_ids': token_type_ids,
            'attention_mask': attention_mask,
            'mlm_labels': mlm_labels,
            'nsp_label': is_next
        })
    
    # Verify NSP balance
    num_pos = sum(1 for ex in processed if ex['nsp_label'] == 1)
    print(f"NSP Balance: {num_pos / len(processed):.1%} positive pairs.")
    
    with open('preprocessed_wikitext2_full.json', 'w') as f:
        json.dump(processed, f)
    print(f"Preprocessing complete. Saved {len(processed)} balanced examples from full WikiText-2-v1.")

if __name__ == "__main__":
    main()


Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Extracted 77684 sentences from full WikiText-2-v1 train split.
NSP Balance: 50.0% positive pairs.
Preprocessing complete. Saved 100000 balanced examples from full WikiText-2-v1.
